# Feature Type Definitions

**`CollectionSchema`** (config class key `CollectionSchema`) is a driver-agnostic schema
decorator that declares which fields are exposed, filtered, or enriched at each
entity level: `catalog`, `collection`, `item`, or `asset`.

It acts as a two-way contract:
- **At storage time** — drivers use it to create backing storage with the right structure
- **At read time** — extensions use it to shape OGC / STAC responses

## Key concepts

| Concept | Purpose |
|---|---|
| `FieldDefinition` | Describes one queryable field: name, data_type, capabilities |
| `FieldCapability` | `filterable`, `sortable`, `groupable`, `aggregatable`, `spatial`, `indexed` |
| `EntityLevel` | `catalog`, `collection`, `item`, `asset` |
| `exclude_fields` | Field names to strip from the driver's native schema |
| `metadata_fields` | Extra metadata to attach (keywords, summaries, etc.) |

## Driver introspection: `get_entity_fields()`

All four drivers (PG, Elasticsearch, Iceberg, DuckDB) implement
`get_entity_fields(catalog_id, collection_id, entity_level="item")`
which returns `Dict[str, FieldDefinition]` — the live schema of the backing store.

In [ ]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv()

BASE = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_TOKEN")
    or ""
)
CATALOG_ID = "demo-feature-type"
COLLECTION_ID = "climate-stations"

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE, headers=headers, timeout=60.0)


def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r


# Clean up any leftover catalog from a previous run before starting
_cleanup = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
if _cleanup.status_code not in (204, 404):
    print(f"WARN: cleanup returned {_cleanup.status_code}: {_cleanup.text[:200]}")

r = client.post("/stac/catalogs", json={
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": "Feature Type Demo",
    "description": "Feature Type Demo catalog.",
    "stac_version": "1.0.0",
})
_check(r, "Catalog")


---
## 1. Declare a feature type at collection creation

Pass `schema` in the collection body.  It is validated as a `CollectionSchema`
and persisted via the config waterfall before the driver's `ensure_storage()`
runs, so the driver can materialise required/unique constraints at DDL time.

In [ ]:
feature_type = {
    "level": "item",
    "fields": {
        "station_id": {
            "name": "station_id",
            "data_type": "string",
            "capabilities": ["filterable", "sortable", "indexed"],
        },
        "temperature": {
            "name": "temperature",
            "data_type": "numeric",
            "capabilities": ["filterable", "sortable", "aggregatable"],
            "aggregations": ["min", "max", "avg"],
        },
        "humidity": {
            "name": "humidity",
            "data_type": "numeric",
            "capabilities": ["filterable", "aggregatable"],
        },
        "geom": {
            "name": "geom",
            "data_type": "geometry",
            "capabilities": ["spatial"],
        },
        "_internal_token": {
            "name": "_internal_token",
            "data_type": "string",
            "expose": False,
            "capabilities": [],
        },
    },
    "exclude_fields": ["raw_payload"],
    "metadata_fields": {
        "keywords": ["climate", "temperature", "humidity", "WMO"],
        "summaries": {"temperature": {"minimum": -50, "maximum": 60}},
    },
}

# Apply catalog-scope driver + routing configs before collection creation.
# CollectionRoutingConfig.operations is Immutable and must exist before POST /collections.
_base = f"/configs/catalogs/{CATALOG_ID}"
_check(client.put(f"{_base}/configs/ItemsPostgresqlDriverConfig", json={
    "enabled": True, "collection_type": "VECTOR",
}), "DriverConfig")
_check(client.put(f"{_base}/configs/CollectionRoutingConfig", json={
    "enabled": True,
    "operations": {
        "WRITE": [{"driver_id": "ItemsPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
        "READ":  [{"driver_id": "ItemsPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
    },
}), "RoutingConfig")

r = client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": "Climate Stations",
    "description": "Climate Stations collection.",
    "license": "CC-BY-4.0",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
    "links": [],
    "schema": feature_type,
})
_check(r, "Collection")


---
## 2. Update feature type via config API

The feature type can be updated independently of the collection metadata.

In [ ]:
# Add a new field after collection already exists
feature_type["fields"]["wind_speed"] = {
    "name": "wind_speed",
    "data_type": "numeric",
    "capabilities": ["filterable", "aggregatable"],
    "aggregations": ["avg", "max"],
}

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionSchema",
    json=feature_type,
)
_check(r, "Feature type updated")


---
## 3. Inspect the stored schema

`GET /configs/catalogs/{cat}/collections/{coll}/configs/CollectionSchema`
returns the `CollectionSchema` config stored at collection scope — the same
object that drivers read at `ensure_storage()` time to build DDL.

To see the *live driver schema* (actual columns / index mappings) use
`GET /features/catalogs/{cat}/collections/{coll}/queryables` — shown in
section 4.

In [ ]:
r = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionSchema",
)
_check(r, "Status")
if r.status_code == 200:
    schema = r.json()
    fields = schema.get("fields", {})
    print(f"\nCollectionSchema reports {len(fields)} declared fields:")
    for name, fd in fields.items():
        caps = ", ".join(fd.get("capabilities", []))
        expose = fd.get("expose", True)
        print(f"  {name:30s} type={fd.get('data_type'):10s} caps=[{caps}]{'  [hidden]' if not expose else ''}")


---
## 4. Queryables — OGC API Features integration

The OGC `/queryables` endpoint uses `introspect_schema()` (which returns
`FieldDefinition` objects) to build the JSON Schema response.  Only fields
with `expose=True` (the default) appear in the public response.

In [ ]:
import random

# Ingest a few items so the schema is populated
features = [
    {
        "type": "Feature",
        "id": f"WMO-{i:05d}",
        "geometry": {"type": "Point", "coordinates": [10 + random.uniform(-20, 20), 45 + random.uniform(-30, 30)]},
        "properties": {
            "datetime": "2024-01-01T00:00:00Z",
            "station_id": f"WMO-{i:05d}",
            "temperature": round(random.uniform(-10, 40), 1),
            "humidity": round(random.uniform(20, 95), 1),
            "wind_speed": round(random.uniform(0, 30), 1),
        },
    }
    for i in range(10)
]
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": features},
)
_check(r, "Items written")

# Queryables — shows only exposed fields
r = client.get(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/queryables")
_check(r, "Queryables status")
if r.status_code == 200:
    q = r.json()
    print(f"Exposed queryable fields: {list(q.get('properties', {}).keys())}")
    # _internal_token should NOT appear (expose=False)


---
## 5. Catalog-scope schema declaration

The same `CollectionSchema` config can be set at **catalog scope** instead
of collection scope.  Catalog-scope acts as a fallback: any collection that
does not set its own `CollectionSchema` inherits the catalog-level one.

Asset-level schemas (`"level": "asset"`) are stored using the same
`CollectionSchema` key.  To separate item-level and asset-level declarations,
store the asset-level schema at catalog scope and the item-level one at
collection scope — the waterfall resolves them independently via the `level`
field.

---
## 6. Field constraints: `required` and `unique`

`FieldDefinition` supports two write-time constraints:

- **`required: true`** — feature is rejected if this field is null/missing.
- **`unique: true`** — the value must be unique within the collection.

Enforcement is **delegated to the primary write driver** via its advertised
capabilities (`REQUIRED_ENFORCEMENT` / `UNIQUE_ENFORCEMENT`). If the primary
driver doesn't support a requested constraint, config admission is rejected
at apply time — unless `allow_app_level_enforcement=true` is set, which opts
into service-layer fallback.

| Driver | `required` | `unique` |
|---|---|---|
| PostgreSQL | ✅ NOT NULL | ✅ UNIQUE (per collection) |
| DuckDB | ✅ NOT NULL | ✅ SQLite UNIQUE |
| Iceberg | ✅ NestedField(required=True) | ⚠️ app-level only |
| Elasticsearch | ⚠️ app-level | ⚠️ app-level |

Secondary async drivers (e.g. ES reindex listeners) consume the *successful*
primary write via the event bus — they never see rejected features, so no
duplicate validation is needed.

In [ ]:
# Mark station_id as required + unique via the config API
constrained = {
    **feature_type,
    "fields": {
        **feature_type["fields"],
        "station_id": {
            "name": "station_id",
            "data_type": "string",
            "capabilities": ["filterable", "sortable", "indexed"],
            "required": True,
            "unique": True,
        },
    },
}

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionSchema",
    json=constrained,
)
_check(r, "Constraints applied")

# Insert a feature missing station_id → rejected (if required enforcement is active)
bad_missing = {
    "type": "Feature", "id": "WMO-BAD-1",
    "geometry": {"type": "Point", "coordinates": [0, 0]},
    "properties": {"datetime": "2024-01-01T00:00:00Z", "temperature": 12.0},  # station_id missing
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": [bad_missing]},
)
_check(r, "Missing required → expect 4xx or 207 rejection")

# Insert a duplicate station_id → rejected (if unique enforcement is active)
dup = {
    "type": "Feature", "id": "WMO-BAD-2",
    "geometry": {"type": "Point", "coordinates": [0, 0]},
    "properties": {"datetime": "2024-01-01T00:00:00Z", "station_id": "WMO-00000", "temperature": 12.0},
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": [dup]},
)
_check(r, "Duplicate unique → expect 4xx or 207 rejection")


In [ ]:
# Declare an asset-level schema at CATALOG scope.
# Stored at catalog scope so it doesn't overwrite the collection-scope item schema.
asset_schema = {
    "level": "asset",
    "fields": {
        "href": {"name": "href", "data_type": "string", "capabilities": ["filterable"]},
        "media_type": {"name": "media_type", "data_type": "string", "capabilities": ["filterable", "groupable"]},
        "file_size": {"name": "file_size", "data_type": "integer", "capabilities": ["filterable", "sortable", "aggregatable"]},
        "checksum": {"name": "checksum", "data_type": "string", "capabilities": [], "expose": False},
    },
}

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/configs/CollectionSchema",
    json=asset_schema,
)
_check(r, "Asset schema stored at catalog scope")


In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
_check(r, "Cleanup")
client.close()
